In [38]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
train_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_real/*.jpg",shuffle=False)
train_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_fake/*.jpg",shuffle=False)
test_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_real/*.jpg",shuffle=False)
test_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_fake/*.jpg",shuffle=False)

In [40]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    return img
train_real = train_real.map(load_image)
train_fake = train_fake.map(load_image)
test_real = test_real.map(load_image)
test_fake = test_fake.map(load_image)

In [41]:
def add_label(image, label):
  return image , label

train_real = train_real.map(lambda x: add_label(x,0))
train_fake = train_fake.map(lambda x: add_label(x,1))
test_real = test_real.map(lambda x: add_label(x, 0))
test_fake = test_fake.map(lambda x: add_label(x, 1))

In [42]:
train_dataset = train_real.concatenate(train_fake)

train_dataset = train_dataset.shuffle(
    buffer_size=7891,
    seed=42,
    reshuffle_each_iteration=False
)

train_size = int(0.8 * 7891)

validation_dataset = train_dataset.skip(train_size)
train_dataset = train_dataset.take(train_size)

In [43]:
def preprocess(image, label):
    image = preprocess_input(image)
    return image, label
train_dataset = train_dataset.map(preprocess)
validation_dataset = validation_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)

In [44]:
BATCH_SIZE = 32
train_dataset = train_dataset.batch(BATCH_SIZE)
validation_dataset = validation_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.batch(BATCH_SIZE)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [45]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

In [46]:
inputs = layers.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.2)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)

In [47]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10
)

Epoch 1/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 664s 3s/step - accuracy: 0.5162 - loss: 0.7007 - val_accuracy: 0.5472 - val_loss: 0.6890
Epoch 2/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 664s 3s/step - accuracy: 0.5344 - loss: 0.6920 - val_accuracy: 0.5769 - val_loss: 0.6808
Epoch 3/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 492s 2s/step - accuracy: 0.5600 - loss: 0.6804 - val_accuracy: 0.6023 - val_loss: 0.6746
Epoch 4/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 615s 3s/step - accuracy: 0.5749 - loss: 0.6753 - val_accuracy: 0.6149 - val_loss: 0.6698
Epoch 5/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 641s 3s/step - accuracy: 0.5919 - loss: 0.6666 - val_accuracy: 0.6276 - val_loss: 0.6655
Epoch 6/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 653s 3s/step - accuracy: 0.5833 - loss: 0.6657 - val_accuracy: 0.6352 - val_loss: 0.6618
Epoch 7/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 592s 3s/step - accuracy: 0.5995 - loss: 0.6640 - val_accuracy: 0.6403 - val_loss: 0.6588
Epoch 8/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 591s 3s/step - accuracy: 0.6120 - loss: 0.6580 - val_accu

In [48]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,053,416 (15.46 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 2,564 (10.02 KB)

In [49]:
model.save("/content/drive/MyDrive/efficientnet_stage2.keras")